In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import torch.nn.functional as F
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

import time

In [ ]:
# --- Check for GPU ---
# This will be important when you move to your cluster
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
#################################################################
# 1. AGENT BRAIN CLASS (As designed)
#################################################################

class AgentBrain(nn.Module):
    """
    An Actor-Critic agent with a shared RNN trunk (GRU).
    """
    def __init__(self, public_input_size, private_input_size, rnn_hidden_size, 
                 move_action_size, pull_action_size, gaze_action_size):
        
        super(AgentBrain, self).__init__()
        
        self.input_size = public_input_size + private_input_size
        self.rnn_hidden_size = rnn_hidden_size
        self.move_action_size = move_action_size
        self.pull_action_size = pull_action_size
        self.gaze_action_size = gaze_action_size

        # --- 1. The Shared RNN "Trunk" ---
        self.rnn_trunk = nn.GRUCell(self.input_size, self.rnn_hidden_size)

        # --- 2. The "Actor" Head ---
        self.move_head = nn.Linear(self.rnn_hidden_size, move_action_size)
        self.pull_head = nn.Linear(self.rnn_hidden_size, pull_action_size)
        self.gaze_head = nn.Linear(self.rnn_hidden_size, gaze_action_size)

        # --- 3. The "Critic" Head ---
        self.critic_head = nn.Linear(self.rnn_hidden_size, 1)

    def forward(self, public_input, private_gaze_input, prev_hidden_state, pull_mask):
        
        # --- Build the Full Input State ---
        full_input = torch.cat([public_input, private_gaze_input], dim=-1)

        # --- Run the "Accumulator" (RNN Trunk) ---
        current_hidden_state = self.rnn_trunk(full_input, prev_hidden_state)

        # --- Get Actor & Critic Outputs ---
        value = self.critic_head(current_hidden_state)
        
        move_logits = self.move_head(current_hidden_state)
        gaze_logits = self.gaze_head(current_hidden_state)
        
        # --- NEW: Apply Action Mask ---
        pull_logits = self.pull_head(current_hidden_state)
        # By adding -inf (or a large negative number), the softmax probability
        # for the masked action (action 1, "pull") becomes zero.
        pull_logits = pull_logits + pull_mask
        # --- END NEW ---
        
        # Return logits, value, and the new hidden state
        return (move_logits, pull_logits, gaze_logits), value, current_hidden_state

### different environment settings

In [ ]:
#################################################################
# 2. MASTER CURRICULUM ENVIRONMENT (FINAL ROBUST VERSION)
#################################################################
class CurriculumEnv:
    def __init__(self):
        self.pull_reset_time = 20
        self.training_phase = "SELF_REWARD" 
        self.current_sync_window = 30 
        
        # --- NEW: Configurable Gaze Cost ---
        # Default is 0.0 so early training is not penalized
        self.gaze_cost = 0.0 
        
        self.reset() 

    def set_training_phase(self, phase, sync_window=30):
        self.training_phase = phase
        self.current_sync_window = sync_window
        print(f"*** ENV UPDATED: Phase={self.training_phase}, SyncWindow={self.current_sync_window}, GazeCost={self.gaze_cost} ***")

    def _get_state(self):
        on_cooldown_A = self.agent_A_pull_cooldown > 0
        is_drinking_A = self.A_is_drinking
        can_pull_A = not (on_cooldown_A or is_drinking_A)
        
        on_cooldown_B = self.agent_B_pull_cooldown > 0
        is_drinking_B = self.B_is_drinking
        can_pull_B = not (on_cooldown_B or is_drinking_B)
            
        state_A = {
            'can_pull': can_pull_A,
            'self_is_drinking': self.A_is_drinking,
            'self_on_cooldown': on_cooldown_A,
            'self_cooldown_remaining': self.agent_A_pull_cooldown,
            'self_drink_timer_remaining': self.A_drink_timer,
            'success_sound': self.success_sound_played 
        }
        state_B = {
            'can_pull': can_pull_B,
            'self_is_drinking': self.B_is_drinking,
            'self_on_cooldown': on_cooldown_B,
            'self_cooldown_remaining': self.agent_B_pull_cooldown,
            'self_drink_timer_remaining': self.B_drink_timer,
            'success_sound': self.success_sound_played
        }
        return state_A, state_B

    def reset(self):
        self.current_step = 0 
        self.pull_A_timestamp = -1 
        self.pull_B_timestamp = -1 
        self.agent_A_pull_cooldown = 0
        self.agent_B_pull_cooldown = 0
        self.success_sound_played = 0
        self.A_time_since_pull = 1000
        self.B_time_since_pull = 1000
        self.A_is_drinking = False
        self.B_is_drinking = False
        self.A_drink_timer = 0
        self.B_drink_timer = 0
        return self._get_state()

    def step(self, action_A, action_B):
        self.current_step += 1
        reward_A, reward_B = -0.01, -0.01 # Base cost of living
        self.success_sound_played = 0 
        done = False
        coordination_attempt_complete = False 

        self.A_time_since_pull += 1
        self.B_time_since_pull += 1

        # --- APPLY VARIABLE GAZE COST ---
        if self.training_phase != "SELF_REWARD":
            if action_A['gaze'] == 1:
                reward_A -= self.gaze_cost
            if action_B['gaze'] == 1:
                reward_B -= self.gaze_cost

        # --- Standard Timer Logic ---
        on_cooldown_A = self.agent_A_pull_cooldown > 0
        is_drinking_A = self.A_is_drinking
        can_pull_A = not (on_cooldown_A or is_drinking_A)
        
        on_cooldown_B = self.agent_B_pull_cooldown > 0
        is_drinking_B = self.B_is_drinking
        can_pull_B = not (on_cooldown_B or is_drinking_B)
        
        if self.A_drink_timer > 0:
            self.A_drink_timer -= 1
            if self.A_drink_timer == 0: self.A_is_drinking = False
        if self.B_drink_timer > 0:
            self.B_drink_timer -= 1
            if self.B_drink_timer == 0: self.B_is_drinking = False
        if self.agent_A_pull_cooldown > 0: self.agent_A_pull_cooldown -= 1
        if self.agent_B_pull_cooldown > 0: self.agent_B_pull_cooldown -= 1

        if self.training_phase == "SELF_REWARD":
            if (action_A['pull'] == 1 and can_pull_A):
                self.A_time_since_pull = 0; reward_A += 15.0; self.success_sound_played = 1
                self.agent_A_pull_cooldown = self.pull_reset_time
                self.A_is_drinking = True; self.A_drink_timer = np.random.randint(30, 50)
            if (action_B['pull'] == 1 and can_pull_B):
                self.B_time_since_pull = 0; reward_B += 15.0; self.success_sound_played = 1
                self.agent_B_pull_cooldown = self.pull_reset_time
                self.B_is_drinking = True; self.B_drink_timer = np.random.randint(30, 50)

        else: # COOP_ASSISTED or COOP_FULL
            if (action_A['pull'] == 1 and can_pull_A):
                self.A_time_since_pull = 0
                self.pull_A_timestamp = self.current_step
                self.agent_A_pull_cooldown = self.pull_reset_time 
                if self.pull_B_timestamp != -1: coordination_attempt_complete = True
            
            if (action_B['pull'] == 1 and can_pull_B):
                self.B_time_since_pull = 0
                self.pull_B_timestamp = self.current_step
                self.agent_B_pull_cooldown = self.pull_reset_time 
                if self.pull_A_timestamp != -1: coordination_attempt_complete = True
            
            if coordination_attempt_complete:
                if abs(self.pull_A_timestamp - self.pull_B_timestamp) <= self.current_sync_window:
                    reward_A += 15.0; reward_B += 15.0
                    self.success_sound_played = 1 
                    self.A_is_drinking = True; self.A_drink_timer = np.random.randint(30, 50)
                    self.B_is_drinking = True; self.B_drink_timer = np.random.randint(30, 50)
                else:
                    if self.training_phase == "COOP_ASSISTED": reward_A += 5.0; reward_B += 5.0 
                    else: reward_A -= 0.5; reward_B -= 0.5
                self.pull_A_timestamp = -1
                self.pull_B_timestamp = -1
        
        next_state_A, next_state_B = self._get_state()
        return next_state_A, next_state_B, reward_A, reward_B, done

### helper functions

In [ ]:
#################################################################
# 3. HELPER FUNCTIONS (NEW - No Travel)
#################################################################

def preprocess_state(state_dict):
    """
    Converts the public state dictionary into a flat tensor.
    --- NO TRAVEL/POSITION INFO ---
    """
    
    # --- Get states ---
    is_drinking = torch.tensor([1.0 if state_dict['self_is_drinking'] else 0.0])
    can_pull = torch.tensor([1.0 if state_dict['can_pull'] else 0.0])
    success_sound = torch.tensor([1.0 if state_dict['success_sound'] else 0.0])
    
    # --- Get normalized timers ---
    # Max drink time is 50, max cooldown is 20
    drink_timer = torch.tensor([state_dict['self_drink_timer_remaining'] / 50.0])
    cooldown_timer = torch.tensor([state_dict['self_cooldown_remaining'] / 20.0])
    
    tensor_state = torch.cat([
        is_drinking,
        can_pull,
        success_sound,
        drink_timer,
        cooldown_timer
    ], dim=0)
    
    return tensor_state.float().unsqueeze(0).to(device) # Add batch dim

# preprocess_partner_pos is no longer needed.

def sample_actions(logits_tuple):
    # (This function is the same, it still samples a 'move' action
    #  which will be ignored by the environment)
    move_logits, pull_logits, gaze_logits = logits_tuple
    move_dist = Categorical(logits=move_logits)
    move_action = move_dist.sample()
    move_log_prob = move_dist.log_prob(move_action)
    pull_dist = Categorical(logits=pull_logits)
    pull_action = pull_dist.sample()
    pull_log_prob = pull_dist.log_prob(pull_action)
    gaze_dist = Categorical(logits=gaze_logits)
    gaze_action = gaze_dist.sample()
    gaze_log_prob = gaze_dist.log_prob(gaze_action)
    total_log_prob = move_log_prob + pull_log_prob + gaze_log_prob
    actions_dict = {
        'move': move_action.item(),
        'pull': pull_action.item(),
        'gaze': gaze_action.item()
    }
    return actions_dict, total_log_prob

### main training function

In [ ]:
#################################################################
# 4. TRAINING HYPERPARAMETERS (NEW)
#################################################################

# --- Environment --
# GRID_X and GRID_Y are removed
SESSION_LENGTH = 200 # Short session for prototyping
EPISODE_LENGTH = 200     # Max steps per trial

# --- Model --
RNN_HIDDEN_SIZE = 128 # 64

# --- NEW INPUT SIZES ---
# is_drinking(1) + can_pull(1) + success_sound(1) + drink_timer(1) + cooldown_timer(1) = 5
PUBLIC_INPUT_SIZE = 5 
# partner_can_pull(1) + partner_time_norm(1) + partner_is_drinking(1) + partner_is_on_cooldown(1) = 4
PRIVATE_INPUT_SIZE = 4

MOVE_ACTIONS = 5  # U, D, L, R, Stay Still (Kept, but ignored by Env)
PULL_ACTIONS = 2  # No, Yes
GAZE_ACTIONS = 2  # No, Yes

# --- Training --
LR = 1e-4                # Learning rate
GAMMA = 0.99             # Discount factor for future rewards
N_STEPS = 500            # (WAS 500) Shorter rollout for faster updates
TOTAL_TRAIN_STEPS = 2200000 
ENTROPY_COEF = 0.0001     
VALUE_LOSS_COEF = 1.0

In [ ]:
#################################################################
# 5. THE MASTER TRAINING LOOP (UPDATED)
#################################################################
def train():
    print("Starting training with FULL 6-PHASE CURRICULUM (No Travel)...\\n") 
    
    env = CurriculumEnv() # No grid args
    
    agent_A = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_B = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)

    optimizer_A = optim.Adam(agent_A.parameters(), lr=LR)
    optimizer_B = optim.Adam(agent_B.parameters(), lr=LR)

    # --- (Curriculum Timings are the same) --
    CURRICULUM_TIMINGS = {
        # --- Phase A: "Always On" (Learn the VALUE of info) --
        0:        ("SELF_REWARD", 200, "ALWAYS_ON"),
        200000:   ("COOP_ASSISTED", 200, "ALWAYS_ON"),
        400000:   ("COOP_FULL", 200, "ALWAYS_ON"),
        500000:   ("COOP_FULL", 50, "ALWAYS_ON"),
        600000:   ("COOP_FULL", 30, "ALWAYS_ON"), # <-- Your "to 50 threshold"
        800000:   ("COOP_FULL", 20, "ALWAYS_ON"),  # <-- Retrain at 50 with filter
        1000000:  ("COOP_FULL", 20, "FILTERED"),
        1200000:  ("COOP_FULL", 15, "FILTERED"),
        1500000:  ("COOP_FULL", 10, "FILTERED")
    }
    
    # --- Get Initial States (Called ONCE) --
    env.set_training_phase("SELF_REWARD", sync_window=200)
    state_A_dict, state_B_dict = env.reset()
    
    public_A = preprocess_state(state_A_dict) # No grid args
    public_B = preprocess_state(state_B_dict) # No grid args
    hidden_A = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    hidden_B = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device) # <-- Size 4
    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device) # <-- Size 4

    # --- (Log init is the same) --
    full_training_log = []
    
    for update_step in range(TOTAL_TRAIN_STEPS // N_STEPS):
        
        total_steps_so_far = update_step * N_STEPS
        
        ## --- (Curriculum update is the same) --
        current_gaze_mode = "FILTERED" # Default
        for milestone_step in sorted(CURRICULUM_TIMINGS.keys(), reverse=True):
            if total_steps_so_far >= milestone_step:
                phase, window, gaze_mode = CURRICULUM_TIMINGS[milestone_step]
                current_gaze_mode = gaze_mode 
                if env.training_phase != phase or env.current_sync_window != window:
                    env.set_training_phase(phase, sync_window=window)
                break
        
        # --- (Rollout init is the same) --
        all_log_probs_A, all_values_A, all_rewards_A = [], [], []
        all_log_probs_B, all_values_B, all_rewards_B = [], [], []
        all_entropy_A, all_entropy_B = 0, 0

        hidden_A = hidden_A.detach()
        hidden_B = hidden_B.detach()

        for step_in_rollout in range(N_STEPS):
            
            # ----------------------------------------------------------
            # --- *** NEW: CREATE ACTION MASKS *** ---
            # ----------------------------------------------------------
            # This logic is now simpler and based *only* on the 'can_pull'
            # state, which the env calculates based on BOTH timers.
            
            if state_A_dict['can_pull']:
                mask_A = torch.tensor([[0., 0.]], device=device).float()
            else:
                mask_A = torch.tensor([[0., -1e9]], device=device).float()
                
            if state_B_dict['can_pull']:
                mask_B = torch.tensor([[0., 0.]], device=device).float()
            else:
                mask_B = torch.tensor([[0., -1e9]], device=device).float()
            # ----------------------------------------------------------
            # --- *** END NEW *** ---
            # ----------------------------------------------------------

            
            # --- (Agent runs now pass the mask) --
            (logits_A_tuple), value_A, next_hidden_A = agent_A(public_A, private_A, hidden_A, mask_A)
            actions_A_dict, log_prob_A = sample_actions(logits_A_tuple)
            entropy_A = -sum((F.softmax(l, dim=-1) * F.log_softmax(l, dim=-1)).sum() for l in logits_A_tuple)
            all_entropy_A += entropy_A
            
            (logits_B_tuple), value_B, next_hidden_B = agent_B(public_B, private_B, hidden_B, mask_B)
            actions_B_dict, log_prob_B = sample_actions(logits_B_tuple)
            entropy_B = -sum((F.softmax(l, dim=-1) * F.log_softmax(l, dim=-1)).sum() for l in logits_B_tuple)
            all_entropy_B += entropy_B

            # --- (Env step is the same) --
            next_state_A_dict, next_state_B_dict, reward_A, reward_B, done = \
                env.step(actions_A_dict, actions_B_dict)
                
            # --- (Logging is modified) --
            log_step = total_steps_so_far + step_in_rollout
            full_training_log.append({
                'step': log_step,
                'phase': env.training_phase,
                'window': env.current_sync_window,
                # 'A_pos_x': env.agent_A_pos[0], # REMOVED
                # 'A_pos_y': env.agent_A_pos[1], # REMOVED
                'A_action_move': actions_A_dict['move'], # Kept, but ignored
                'A_action_pull': actions_A_dict['pull'],
                'A_action_gaze': actions_A_dict['gaze'],
                'A_reward': reward_A, 
                'A_value': value_A.item(),
                # 'B_pos_x': env.agent_B_pos[0], # REMOVED
                # 'B_pos_y': env.agent_B_pos[1], # REMOVED
                'B_action_move': actions_B_dict['move'], # Kept, but ignored
                'B_action_pull': actions_B_dict['pull'],
                'B_action_gaze': actions_B_dict['gaze'],
                'B_reward': reward_B, 
                'B_value': value_B.item(),
            })

            # --- (Storing rewards/logs is the same) --
            all_log_probs_A.append(log_prob_A)
            all_values_A.append(value_A)
            all_rewards_A.append(reward_A)
            all_log_probs_B.append(log_prob_B)
            all_values_B.append(value_B)
            all_rewards_B.append(reward_B)

            # --- *** NEW: Gaze Tensor Logic *** ---
            # (No position, 4 features)
            
            # --- Partner B's state for Agent A's gaze ---
            partner_B_can_pull = torch.tensor([[
                1.0 if next_state_B_dict['can_pull'] else 0.0
            ]], device=device).float()
            partner_B_time_norm = torch.tensor([[
                min(env.B_time_since_pull / env.current_sync_window, 1.0)
            ]], device=device).float()
            partner_B_is_drinking = torch.tensor([[
                1.0 if next_state_B_dict['self_is_drinking'] else 0.0
            ]], device=device).float()
            partner_B_is_on_cooldown = torch.tensor([[
                1.0 if next_state_B_dict['self_on_cooldown'] else 0.0
            ]], device=device).float()

            # --- Partner A's state for Agent B's gaze ---
            partner_A_can_pull = torch.tensor([[
                1.0 if next_state_A_dict['can_pull'] else 0.0
            ]], device=device).float()
            partner_A_time_norm = torch.tensor([[
                min(env.A_time_since_pull / env.current_sync_window, 1.0)
            ]], device=device).float()
            partner_A_is_drinking = torch.tensor([[
                1.0 if next_state_A_dict['self_is_drinking'] else 0.0
            ]], device=device).float()
            partner_A_is_on_cooldown = torch.tensor([[
                1.0 if next_state_A_dict['self_on_cooldown'] else 0.0
            ]], device=device).float()


            if current_gaze_mode == "ALWAYS_ON":
                # Agents get full info for free, regardless of gaze action
                private_A = torch.cat([partner_B_can_pull, partner_B_time_norm, partner_B_is_drinking, partner_B_is_on_cooldown], dim=-1)
                private_B = torch.cat([partner_A_can_pull, partner_A_time_norm, partner_A_is_drinking, partner_A_is_on_cooldown], dim=-1)
            
            else: # "FILTERED" mode
                if actions_A_dict['gaze'] == 1:
                    private_A = torch.cat([partner_B_can_pull, partner_B_time_norm, partner_B_is_drinking, partner_B_is_on_cooldown], dim=-1)
                else:
                    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device) 
                
                if actions_B_dict['gaze'] == 1:
                    private_B = torch.cat([partner_A_can_pull, partner_A_time_norm, partner_A_is_drinking, partner_A_is_on_cooldown], dim=-1)
                else:
                    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
            # --- (End Gaze Tensor logic) ---


            public_A = preprocess_state(next_state_A_dict) # No grid args
            public_B = preprocess_state(next_state_B_dict) # No grid args
            hidden_A = next_hidden_A
            hidden_B = next_hidden_B

            state_A_dict = next_state_A_dict
            state_B_dict = next_state_B_dict

        # --- (Learning/Backprop is the same) --
        # Note: mask_A and mask_B are the *last* masks from the loop
        _, next_value_A, _ = agent_A(public_A, private_A, hidden_A, mask_A) 
        _, next_value_B, _ = agent_B(public_B, private_B, hidden_B, mask_B) 

        rewards_A = torch.tensor(all_rewards_A).float().to(device)
        values_A = torch.cat(all_values_A)
        log_probs_A = torch.cat(all_log_probs_A)
        returns_A = torch.zeros_like(rewards_A)
        R_A = next_value_A.detach()
        for t in reversed(range(len(rewards_A))):
            R_A = rewards_A[t] + GAMMA * R_A
            returns_A[t] = R_A
        advantages_A = returns_A - values_A.detach() 
        actor_loss_A = -(log_probs_A * advantages_A).mean()
        critic_loss_A = F.mse_loss(values_A.squeeze(), returns_A)
        loss_A = (actor_loss_A + VALUE_LOSS_COEF * critic_loss_A - ENTROPY_COEF * all_entropy_A)

        rewards_B = torch.tensor(all_rewards_B).float().to(device)
        values_B = torch.cat(all_values_B)
        log_probs_B = torch.cat(all_log_probs_B)
        returns_B = torch.zeros_like(rewards_B)
        R_B = next_value_B.detach()
        for t in reversed(range(len(rewards_B))):
            R_B = rewards_B[t] + GAMMA * R_B
            returns_B[t] = R_B
        advantages_B = returns_B - values_B.detach()
        actor_loss_B = -(log_probs_B * advantages_B).mean()
        critic_loss_B = F.mse_loss(values_B.squeeze(), returns_B)
        loss_B = (actor_loss_B + VALUE_LOSS_COEF * critic_loss_B - ENTROPY_COEF * all_entropy_B)

        # --- (Optimization step is the same) --
        optimizer_A.zero_grad()
        loss_A.backward()
        torch.nn.utils.clip_grad_norm_(agent_A.parameters(), 1.0)
        optimizer_A.step()
        optimizer_B.zero_grad()
        loss_B.backward()
        torch.nn.utils.clip_grad_norm_(agent_B.parameters(), 1.0)
        optimizer_B.step()
        
        # --- (Printing is the same) --
        if update_step % (5000 // N_STEPS) == 0: 
            total_steps_so_far = update_step * N_STEPS
            total_rewards_A = sum(all_rewards_A)
            total_rewards_B = sum(all_rewards_B)
            print(f"Step: {total_steps_so_far}, Phase: {env.training_phase}, Win: {env.current_sync_window}, "
                  f"R_A: {total_rewards_A:.2f}, R_B: {total_rewards_B:.2f}, "
                  f"Loss_A: {loss_A.item():.4f}, Loss_B: {loss_B.item():.4f}")

    print("Training complete.")
    
    # --- (Saving log is the same) --
    print("Converting log to DataFrame...")
    df = pd.DataFrame(full_training_log)
    df.to_pickle("full_training_log_new_new.pkl")
    print("Full training log saved to 'full_training_log_new_new.pkl'")
    
    torch.save(agent_A.state_dict(), "agent_A_brain_new_new.pth")
    torch.save(agent_B.state_dict(), "agent_B_brain_new_new.pth")
    

# --- To run the training --
if 0:
    if __name__ == "__main__":
        train()
else:
    print('no training over all')



In [ ]:
#################################################################
# 7. CONTINUE TRAINING (Fine-Tune at 1s Threshold - Window 10)
#    - NOW SAVES THE TRAINING LOG -
#################################################################

def continue_training_1s():
    """
    Loads the trained "new_new" models and continues training
    specifically at the 1s threshold (Window=10) with FILTERED gaze.
    """
    # --- Configuration ---
    CONTINUE_STEPS = 500000  
    TARGET_WINDOW = 10       # 10 steps = 1.0 seconds
    GAZE_MODE = "FILTERED"   # <--- Explicitly set to FILTERED
    LR_FINE_TUNE = 5e-5      
    
    print(f"--- Resuming Training ---")
    print(f"   Window: {TARGET_WINDOW} steps (1.0s)")
    print(f"   Gaze Mode: {GAZE_MODE} (Agents must gaze to see partner)")
    print(f"   Steps: {CONTINUE_STEPS}\n")
    
    # --- 2. Setup Environment ---
    env = CurriculumEnv()
    env.set_training_phase("COOP_FULL", sync_window=TARGET_WINDOW) 
    
    # --- 3. Initialize Agents ---
    agent_A = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_B = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)

    # --- 4. Load Previous Weights ---
    try:
        agent_A.load_state_dict(torch.load("agent_A_brain_new_new.pth"))
        agent_B.load_state_dict(torch.load("agent_B_brain_new_new.pth"))
        print("Successfully loaded 'agent_A/B_brain_new_new.pth'")
    except FileNotFoundError:
        print("Error: Could not find saved models! Please run train() first.")
        return

    # --- 5. Setup Optimizers ---
    optimizer_A = optim.Adam(agent_A.parameters(), lr=LR_FINE_TUNE)
    optimizer_B = optim.Adam(agent_B.parameters(), lr=LR_FINE_TUNE)

    # --- 6. Initialize State vars ---
    state_A_dict, state_B_dict = env.reset()
    public_A = preprocess_state(state_A_dict)
    public_B = preprocess_state(state_B_dict)
    hidden_A = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    hidden_B = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)

    # --- 7. Training Loop ---
    fine_tune_log = []
    
    for update_step in range(CONTINUE_STEPS // N_STEPS):
        
        total_steps_so_far = update_step * N_STEPS
        
        # Reset buffers
        all_log_probs_A, all_values_A, all_rewards_A = [], [], []
        all_log_probs_B, all_values_B, all_rewards_B = [], [], []
        all_entropy_A, all_entropy_B = 0, 0
        
        hidden_A = hidden_A.detach()
        hidden_B = hidden_B.detach()

        # --- Rollout ---
        for step in range(N_STEPS):
            
            # Masks 
            if state_A_dict['can_pull']:
                mask_A = torch.tensor([[0., 0.]], device=device).float()
            else:
                mask_A = torch.tensor([[0., -1e9]], device=device).float()
            if state_B_dict['can_pull']:
                mask_B = torch.tensor([[0., 0.]], device=device).float()
            else:
                mask_B = torch.tensor([[0., -1e9]], device=device).float()

            # Agent steps
            (logits_A_tuple), value_A, next_hidden_A = agent_A(public_A, private_A, hidden_A, mask_A)
            actions_A_dict, log_prob_A = sample_actions(logits_A_tuple) 
            
            (logits_B_tuple), value_B, next_hidden_B = agent_B(public_B, private_B, hidden_B, mask_B)
            actions_B_dict, log_prob_B = sample_actions(logits_B_tuple)

            # Entropy
            entropy_A = -sum((F.softmax(l, dim=-1) * F.log_softmax(l, dim=-1)).sum() for l in logits_A_tuple)
            all_entropy_A += entropy_A
            entropy_B = -sum((F.softmax(l, dim=-1) * F.log_softmax(l, dim=-1)).sum() for l in logits_B_tuple)
            all_entropy_B += entropy_B

            # Env Step
            next_state_A_dict, next_state_B_dict, reward_A, reward_B, done = \
                env.step(actions_A_dict, actions_B_dict)
            
            # Logging
            fine_tune_log.append({
                'step': total_steps_so_far + step,
                'A_reward': reward_A,
                'B_reward': reward_B,
                'A_action_pull': actions_A_dict['pull'],
                'B_action_pull': actions_B_dict['pull'],
                'window': env.current_sync_window
            })

            all_log_probs_A.append(log_prob_A); all_values_A.append(value_A); all_rewards_A.append(reward_A)
            all_log_probs_B.append(log_prob_B); all_values_B.append(value_B); all_rewards_B.append(reward_B)

            # --- Gaze Logic (Using GAZE_MODE) ---
            
            # 1. Create the tensors first
            pB_can = torch.tensor([[1.0 if next_state_B_dict['can_pull'] else 0.0]], device=device).float()
            pB_time = torch.tensor([[min(env.B_time_since_pull / env.current_sync_window, 1.0)]], device=device).float()
            pB_drink = torch.tensor([[1.0 if next_state_B_dict['self_is_drinking'] else 0.0]], device=device).float()
            pB_cool = torch.tensor([[1.0 if next_state_B_dict['self_on_cooldown'] else 0.0]], device=device).float()
            
            pA_can = torch.tensor([[1.0 if next_state_A_dict['can_pull'] else 0.0]], device=device).float()
            pA_time = torch.tensor([[min(env.A_time_since_pull / env.current_sync_window, 1.0)]], device=device).float()
            pA_drink = torch.tensor([[1.0 if next_state_A_dict['self_is_drinking'] else 0.0]], device=device).float()
            pA_cool = torch.tensor([[1.0 if next_state_A_dict['self_on_cooldown'] else 0.0]], device=device).float()

            # 2. Apply the Logic
            if GAZE_MODE == "ALWAYS_ON":
                private_A = torch.cat([pB_can, pB_time, pB_drink, pB_cool], dim=-1)
                private_B = torch.cat([pA_can, pA_time, pA_drink, pA_cool], dim=-1)
            else: # "FILTERED"
                if actions_A_dict['gaze'] == 1:
                    private_A = torch.cat([pB_can, pB_time, pB_drink, pB_cool], dim=-1)
                else:
                    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
                    
                if actions_B_dict['gaze'] == 1:
                    private_B = torch.cat([pA_can, pA_time, pA_drink, pA_cool], dim=-1)
                else:
                    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)

            # Update vars
            public_A = preprocess_state(next_state_A_dict)
            public_B = preprocess_state(next_state_B_dict)
            hidden_A = next_hidden_A
            hidden_B = next_hidden_B
            state_A_dict = next_state_A_dict
            state_B_dict = next_state_B_dict

        # --- Update (PPO / A2C style) ---
        # Agent A
        _, next_val_A, _ = agent_A(public_A, private_A, hidden_A, mask_A)
        rewards_A_t = torch.tensor(all_rewards_A).float().to(device)
        values_A_t = torch.cat(all_values_A)
        log_probs_A_t = torch.cat(all_log_probs_A)
        returns_A = torch.zeros_like(rewards_A_t)
        R_A = next_val_A.detach()
        for t in reversed(range(len(rewards_A_t))):
            R_A = rewards_A_t[t] + GAMMA * R_A
            returns_A[t] = R_A
        adv_A = returns_A - values_A_t.detach()
        loss_A = -(log_probs_A_t * adv_A).mean() + VALUE_LOSS_COEF * F.mse_loss(values_A_t.squeeze(), returns_A) - ENTROPY_COEF * all_entropy_A

        # Agent B
        _, next_val_B, _ = agent_B(public_B, private_B, hidden_B, mask_B)
        rewards_B_t = torch.tensor(all_rewards_B).float().to(device)
        values_B_t = torch.cat(all_values_B)
        log_probs_B_t = torch.cat(all_log_probs_B)
        returns_B = torch.zeros_like(rewards_B_t)
        R_B = next_val_B.detach()
        for t in reversed(range(len(rewards_B_t))):
            R_B = rewards_B_t[t] + GAMMA * R_B
            returns_B[t] = R_B
        adv_B = returns_B - values_B_t.detach()
        loss_B = -(log_probs_B_t * adv_B).mean() + VALUE_LOSS_COEF * F.mse_loss(values_B_t.squeeze(), returns_B) - ENTROPY_COEF * all_entropy_B

        # Backprop
        optimizer_A.zero_grad()
        loss_A.backward()
        torch.nn.utils.clip_grad_norm_(agent_A.parameters(), 1.0)
        optimizer_A.step()

        optimizer_B.zero_grad()
        loss_B.backward()
        torch.nn.utils.clip_grad_norm_(agent_B.parameters(), 1.0)
        optimizer_B.step()

        if update_step % 10 == 0:
            print(f"Fine-Tune Step {total_steps_so_far}: R_A={sum(all_rewards_A):.2f}, R_B={sum(all_rewards_B):.2f}")

    print("Fine-tuning complete.")
    
    # --- 8. Save the "1s Specialist" Models ---
    torch.save(agent_A.state_dict(), "agent_A_1s_expert.pth")
    torch.save(agent_B.state_dict(), "agent_B_1s_expert.pth")
    print("Models saved as 'agent_A_1s_expert.pth' and 'agent_B_1s_expert.pth'")
    
    # --- 9. Save Training Log ---
    print("Saving fine-tuning log...")
    df_fine_tune = pd.DataFrame(fine_tune_log)
    df_fine_tune.to_pickle("fine_tune_log_1s.pkl")
    print("Fine-tuning log saved to 'fine_tune_log_1s.pkl'")

# --- Run it ---
if 0:
    if __name__ == "__main__":
        continue_training_1s()
else:
    print('no 1s fine tune training')

        
        

In [ ]:
# combine the two logs
# --- 1. Load the two log files ---
try:
    df_train = pd.read_pickle("full_training_log_new_new.pkl")
    df_finetune = pd.read_pickle("fine_tune_log_1s.pkl")
    print(f"Loaded Training Log: {len(df_train)} steps")
    print(f"Loaded Fine-Tune Log: {len(df_finetune)} steps")
except FileNotFoundError as e:
    print(f"Error loading files: {e}")
    print("Make sure both .pkl files are in the current directory.")

# --- 2. Adjust the "Step" Count ---
# The fine-tune log likely starts at step 0 or relative to the start of fine-tuning.
# We need to shift it so it starts exactly where the training log ended.

last_step = df_train['step'].iloc[-1]
print(f"Training ended at step: {last_step}")

# Shift the fine-tune steps
# Note: We add +1 to ensure no overlap, or just add the last step count
df_finetune['step'] = df_finetune['step'] + last_step + 1

# --- 3. Standardize Columns (Handle Missing Data) ---
# The fine-tune log might be missing columns like 'phase' or 'A_action_gaze' 
# depending on exactly which version of the code you ran.

# Set a distinct phase name for the plot
df_finetune['phase'] = "FINE_TUNE_1S"

# Ensure essential columns exist (fill with 0 or NaN if missing)
for col in df_train.columns:
    if col not in df_finetune.columns:
        df_finetune[col] = None # or 0

# --- 4. Concatenate ---
df_combined = pd.concat([df_train, df_finetune], ignore_index=True)
print(f"Combined Log Length: {len(df_combined)} steps")

df = df_combined

In [ ]:
np.unique(df['A_reward'])

In [ ]:
plt.plot(df['A_action_pull'][0:1000])

In [ ]:
print("Calculating pull efficiency...")
import seaborn as sns


# Use the 100-step bin as requested
bin_size = 100
df['step_bin'] = (df['step'] // bin_size) * bin_size

# Define "attempted" vs "successful" pulls
# A "successful" pull is one that resulted in a large reward (> 5.0)
df['A_attempted_pull'] = (df['A_action_pull'] == 1)
df['B_attempted_pull'] = (df['B_action_pull'] == 1)
df['A_successful_pull'] = (df['A_action_pull'] == 1) & (df['A_reward'] > 5.0)
df['B_successful_pull'] = (df['B_action_pull'] == 1) & (df['B_reward'] > 5.0)


# --- 3. Aggregate Data ---
# Group by the 100-step bins and sum up the attempts and successes
df_agg = df.groupby('step_bin').agg(
    A_attempted_pulls=('A_attempted_pull', 'sum'),
    A_successful_pulls=('A_successful_pull', 'sum'),
    B_attempted_pulls=('B_attempted_pull', 'sum'),
    B_successful_pulls=('B_successful_pull', 'sum'),
    phase=('phase', 'first'),
    window=('window', 'first')
).reset_index()

# --- 4. Calculate Success Rate (Pull Efficiency) ---
# Calculate the raw success rate for each bin
# We use .fillna(0) to handle bins with 0 attempts (0/0 = NaN)
df_agg['A_success_rate'] = (df_agg['A_successful_pulls'] / df_agg['A_attempted_pulls']).fillna(0)
df_agg['B_success_rate'] = (df_agg['B_successful_pulls'] / df_agg['B_attempted_pulls']).fillna(0)

# --- 5. Smooth the Data (Rolling Average) ---
# The raw 100-step rate is too noisy. Let's smooth it with a 10,000-step
# (100 bins) rolling average to see the trend.
rolling_window_size = 100 
df_agg['A_rate_rolling'] = df_agg['A_success_rate'].rolling(window=rolling_window_size, min_periods=1).mean()
df_agg['B_rate_rolling'] = df_agg['B_success_rate'].rolling(window=rolling_window_size, min_periods=1).mean()

print("Calculation complete. Plotting...")

# --- 6. Plotting ---
plt.figure(figsize=(20, 10))
sns.lineplot(data=df_agg, x='step_bin', y='A_rate_rolling', label='Agent A: Pull Efficiency (10k-step avg)')
sns.lineplot(data=df_agg, x='step_bin', y='B_rate_rolling', label='Agent B: Pull Efficiency (10k-step avg)')

plt.title('Pull Efficiency Over Training (Successful Pulls / Total Attempted Pulls)', fontsize=18)
plt.ylabel('Pull Efficiency (Rolling Average)', fontsize=14)
plt.xlabel('Training Step', fontsize=14)
plt.legend(loc='upper left', fontsize=12)

# --- Add vertical lines for curriculum changes ---
phase_changes = df_agg.drop_duplicates('phase')[['step_bin', 'phase']].iloc[1:]
window_changes = df_agg.drop_duplicates('window')[['step_bin', 'window']].iloc[1:]

ax = plt.gca()
for _, row in phase_changes.iterrows():
    ax.axvline(x=row['step_bin'], color='red', linestyle='--', linewidth=2)
    ax.text(row['step_bin'] + 5000, 0.095, f"Phase: {row['phase']}", 
            color='red', rotation=90, ha='left', va='top', fontsize=12)

for _, row in window_changes.iterrows():
    ax.axvline(x=row['step_bin'], color='blue', linestyle=':', linewidth=2)
    ax.text(row['step_bin'] + 5000, 0.07, f"Window: {row['window']}", 
            color='blue', rotation=90, ha='left', va='top', fontsize=12)

plt.ylim(0, 1) # Efficiency is a rate from 0 to 1
plt.savefig("pull_efficiency_plot.pdf")

In [ ]:
# Set plotting style
sns.set(style="darkgrid")
plt.rcParams['figure.figsize'] = (18, 8)


# --- 2. Get Curriculum Change Points ---
# We do this first, on the full dataframe
bin_size = 5000
df['step_bin'] = (df['step'] // bin_size) * bin_size
df_agg_bins = df.groupby('step_bin').first().reset_index()
phase_changes = df_agg_bins.drop_duplicates('phase')[['step', 'phase']].iloc[1:]
window_changes = df_agg_bins.drop_duplicates('window')[['step', 'window']].iloc[1:]


# --- 3. Feature Engineering: Find Successful Pulls ---
print("Calculating Inter-Pull Intervals (IPI)...")

# A "successful" pull is one that resulted in a large reward (> 5.0)
df['A_successful_pull'] = (df['A_action_pull'] == 1)  & (df['A_reward'] > 5.0)
df['B_successful_pull'] = (df['B_action_pull'] == 1)  & (df['B_reward'] > 5.0)

# --- 4. Calculate IPI ---
# Create new, "sparse" DataFrames containing only the successful pulls
df_A_pulls = df[df['A_successful_pull']].copy()
df_B_pulls = df[df['B_successful_pull']].copy()

# Calculate the difference (in steps) between each successful pull
df_A_pulls['A_IPI'] = df_A_pulls['step'].diff()
df_B_pulls['B_IPI'] = df_B_pulls['step'].diff()

# --- 5. Smooth the Data (Rolling Average) ---
# Calculate the rolling average of the last 50 IPIs to see the trend
rolling_window_pulls = 50
df_A_pulls['A_IPI_rolling'] = df_A_pulls['A_IPI'].rolling(window=rolling_window_pulls, min_periods=1).mean()
df_B_pulls['B_IPI_rolling'] = df_B_pulls['B_IPI'].rolling(window=rolling_window_pulls, min_periods=1).mean()

print("Calculation complete. Plotting...")

# --- 6. Plotting ---
plt.figure(figsize=(20, 10))
sns.lineplot(data=df_A_pulls, x='step', y='A_IPI_rolling', label=f'Agent A: Avg. IPI (last {rolling_window_pulls} pulls)')
sns.lineplot(data=df_B_pulls, x='step', y='B_IPI_rolling', label=f'Agent B: Avg. IPI (last {rolling_window_pulls} pulls)')

# Add a line for the "optimal" pull interval (60-step cooldown + 5-step personal)
plt.axhline(y=65, color='gray', linestyle=':', linewidth=2, label='Optimal IPI (65 steps)')

plt.title('Inter-Pull Interval (IPI) Over Training', fontsize=18)
plt.ylabel('Average Interval (steps)', fontsize=14)
plt.xlabel('Training Step', fontsize=14)
plt.legend(loc='upper left', fontsize=12)
plt.ylim(0,400) # Zoom in on a reasonable IPI range

# --- Add vertical lines for curriculum changes ---
ax = plt.gca()
for _, row in phase_changes.iterrows():
    ax.axvline(x=row['step'], color='red', linestyle='--', linewidth=2)
    ax.text(row['step'] + 5000, 390, f"Phase: {row['phase']}", 
            color='red', rotation=90, ha='left', va='top', fontsize=12)

for _, row in window_changes.iterrows():
    ax.axvline(x=row['step'], color='blue', linestyle=':', linewidth=2)
    ax.text(row['step'] + 5000, 340, f"Window: {row['window']}", 
            color='blue', rotation=90, ha='left', va='top', fontsize=12)

plt.savefig("ipi_plot.pdf")
print("Plot saved as 'ipi_plot.pdf")

### train with gaze with different social evidence strength (modelled as the gaze reliability)

In [ ]:
#################################################################
# 7. EXPERIMENT: TRAINING A "FLEXIBLE" GENERALIST AGENT
#    (Randomized Gaze Reliability)
#################################################################

def continue_training_generalist():
    """
    Fine-tunes the baseline expert on a MIX of gaze reliabilities.
    The agent must learn to adapt its strategy to current noise levels.
    """
    # --- Configuration ---
    CONTINUE_STEPS = 1000000  # Needs more time to learn flexibility (1M steps)
    TARGET_WINDOW = 10       
    LR_FINE_TUNE = 5e-5
    
    print(f"\n--- Starting GENERALIST Training ---")
    print("The agent will encounter random gaze reliability (10% to 100%)")
    print("Goal: Learn to gaze MORE when reliability is LOW.\n")
    
    # --- 2. Setup Environment ---
    env = CurriculumEnv()
    env.set_training_phase("COOP_FULL", sync_window=TARGET_WINDOW)
    
    # *** ACTIVATE THE COST HERE ***
    env.gaze_cost = 0.03 
    print(f"   Gaze Cost Active: {env.gaze_cost}")
    
    
    # --- 3. Initialize Agents ---
    agent_A = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_B = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)

    # --- 4. Load the 1s Expert (Best Baseline) ---
    try:
        agent_A.load_state_dict(torch.load("agent_A_1s_expert.pth"))
        agent_B.load_state_dict(torch.load("agent_B_1s_expert.pth"))
        print("Loaded '1s Expert' weights.")
    except FileNotFoundError:
        print("Error: No expert models found! Train the 1s expert first.")
        return

    optimizer_A = optim.Adam(agent_A.parameters(), lr=LR_FINE_TUNE)
    optimizer_B = optim.Adam(agent_B.parameters(), lr=LR_FINE_TUNE)

    # --- 6. Initialize State vars ---
    state_A_dict, state_B_dict = env.reset()
    public_A = preprocess_state(state_A_dict)
    public_B = preprocess_state(state_B_dict)
    hidden_A = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    hidden_B = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)

    fine_tune_log = []
    
    # --- Initialize Reliability ---
    current_reliability = 1.0 
    
    for update_step in range(CONTINUE_STEPS // N_STEPS):
        total_steps_so_far = update_step * N_STEPS
        
        # --- DYNAMIC RELIABILITY ---
        # Every 5 updates (2500 steps), pick a new reliability between 0.1 and 1.0
        if update_step % 5 == 0:
            current_reliability = np.random.uniform(0.1, 1.0)
        
        all_log_probs_A, all_values_A, all_rewards_A = [], [], []
        all_log_probs_B, all_values_B, all_rewards_B = [], [], []
        all_entropy_A, all_entropy_B = 0, 0
        
        hidden_A = hidden_A.detach()
        hidden_B = hidden_B.detach()

        for step in range(N_STEPS):
            
            # Masks 
            if state_A_dict['can_pull']:
                mask_A = torch.tensor([[0., 0.]], device=device).float()
            else:
                mask_A = torch.tensor([[0., -1e9]], device=device).float()
            if state_B_dict['can_pull']:
                mask_B = torch.tensor([[0., 0.]], device=device).float()
            else:
                mask_B = torch.tensor([[0., -1e9]], device=device).float()

            # Agent steps
            (logits_A_tuple), value_A, next_hidden_A = agent_A(public_A, private_A, hidden_A, mask_A)
            actions_A_dict, log_prob_A = sample_actions(logits_A_tuple) 
            (logits_B_tuple), value_B, next_hidden_B = agent_B(public_B, private_B, hidden_B, mask_B)
            actions_B_dict, log_prob_B = sample_actions(logits_B_tuple)

            # Entropy
            entropy_A = -sum((F.softmax(l, dim=-1) * F.log_softmax(l, dim=-1)).sum() for l in logits_A_tuple)
            all_entropy_A += entropy_A
            entropy_B = -sum((F.softmax(l, dim=-1) * F.log_softmax(l, dim=-1)).sum() for l in logits_B_tuple)
            all_entropy_B += entropy_B

            # Env Step
            next_state_A_dict, next_state_B_dict, reward_A, reward_B, done = \
                env.step(actions_A_dict, actions_B_dict)
            
            # --- LOGGING (CORRECTED) ---
            fine_tune_log.append({
                'step': total_steps_so_far + step,
                'reliability': current_reliability, 
                'A_reward': reward_A,
                'B_reward': reward_B,
                'A_action_pull': actions_A_dict['pull'], # <--- ADDED
                'B_action_pull': actions_B_dict['pull'], # <--- ADDED
                'A_action_gaze': actions_A_dict['gaze'],
                'B_action_gaze': actions_B_dict['gaze']
            })

            all_log_probs_A.append(log_prob_A); all_values_A.append(value_A); all_rewards_A.append(reward_A)
            all_log_probs_B.append(log_prob_B); all_values_B.append(value_B); all_rewards_B.append(reward_B)

            # -------------------------------------------------------
            # --- DYNAMIC RELIABILITY GAZE LOGIC ---
            # -------------------------------------------------------
            
            # 1. Truth
            pB_can = torch.tensor([[1.0 if next_state_B_dict['can_pull'] else 0.0]], device=device).float()
            pB_time = torch.tensor([[min(env.B_time_since_pull / env.current_sync_window, 1.0)]], device=device).float()
            pB_drink = torch.tensor([[1.0 if next_state_B_dict['self_is_drinking'] else 0.0]], device=device).float()
            pB_cool = torch.tensor([[1.0 if next_state_B_dict['self_on_cooldown'] else 0.0]], device=device).float()
            
            pA_can = torch.tensor([[1.0 if next_state_A_dict['can_pull'] else 0.0]], device=device).float()
            pA_time = torch.tensor([[min(env.A_time_since_pull / env.current_sync_window, 1.0)]], device=device).float()
            pA_drink = torch.tensor([[1.0 if next_state_A_dict['self_is_drinking'] else 0.0]], device=device).float()
            pA_cool = torch.tensor([[1.0 if next_state_A_dict['self_on_cooldown'] else 0.0]], device=device).float()

            # 2. Apply Mask based on CURRENT randomized reliability
            gaze_mask_A = (torch.rand(1, device=device) < current_reliability).float()
            gaze_mask_B = (torch.rand(1, device=device) < current_reliability).float()

            if actions_A_dict['gaze'] == 1:
                private_A = torch.cat([pB_can, pB_time, pB_drink, pB_cool], dim=-1) * gaze_mask_A
            else:
                private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
                
            if actions_B_dict['gaze'] == 1:
                private_B = torch.cat([pA_can, pA_time, pA_drink, pA_cool], dim=-1) * gaze_mask_B
            else:
                private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)

            # Update vars
            public_A = preprocess_state(next_state_A_dict)
            public_B = preprocess_state(next_state_B_dict)
            hidden_A = next_hidden_A
            hidden_B = next_hidden_B
            state_A_dict = next_state_A_dict
            state_B_dict = next_state_B_dict

        # --- Update ---
        _, next_val_A, _ = agent_A(public_A, private_A, hidden_A, mask_A)
        rewards_A_t = torch.tensor(all_rewards_A).float().to(device)
        values_A_t = torch.cat(all_values_A)
        log_probs_A_t = torch.cat(all_log_probs_A)
        returns_A = torch.zeros_like(rewards_A_t)
        R_A = next_val_A.detach()
        for t in reversed(range(len(rewards_A_t))):
            R_A = rewards_A_t[t] + GAMMA * R_A
            returns_A[t] = R_A
        adv_A = returns_A - values_A_t.detach()
        loss_A = -(log_probs_A_t * adv_A).mean() + VALUE_LOSS_COEF * F.mse_loss(values_A_t.squeeze(), returns_A) - ENTROPY_COEF * all_entropy_A

        _, next_val_B, _ = agent_B(public_B, private_B, hidden_B, mask_B)
        rewards_B_t = torch.tensor(all_rewards_B).float().to(device)
        values_B_t = torch.cat(all_values_B)
        log_probs_B_t = torch.cat(all_log_probs_B)
        returns_B = torch.zeros_like(rewards_B_t)
        R_B = next_val_B.detach()
        for t in reversed(range(len(rewards_B_t))):
            R_B = rewards_B_t[t] + GAMMA * R_B
            returns_B[t] = R_B
        adv_B = returns_B - values_B_t.detach()
        loss_B = -(log_probs_B_t * adv_B).mean() + VALUE_LOSS_COEF * F.mse_loss(values_B_t.squeeze(), returns_B) - ENTROPY_COEF * all_entropy_B

        optimizer_A.zero_grad(); loss_A.backward(); torch.nn.utils.clip_grad_norm_(agent_A.parameters(), 1.0); optimizer_A.step()
        optimizer_B.zero_grad(); loss_B.backward(); torch.nn.utils.clip_grad_norm_(agent_B.parameters(), 1.0); optimizer_B.step()

        if update_step % 20 == 0:
            # Now explicitly showing losses in the print statement
            print(f"Gen. Training Step {total_steps_so_far}: "
                  f"Rel={current_reliability:.2f}, "
                  f"R_A={sum(all_rewards_A):.1f}, R_B={sum(all_rewards_B):.1f}, "
                  f"L_A={loss_A.item():.4f}, L_B={loss_B.item():.4f}")

    # --- Save ---
    torch.save(agent_A.state_dict(), "agent_A_generalist.pth")
    torch.save(agent_B.state_dict(), "agent_B_generalist.pth")
    
    df_log = pd.DataFrame(fine_tune_log)
    df_log.to_pickle("log_generalist.pkl")
    print("Saved generalist models and log.")

    
# --- Run ---
if 1:
    if __name__ == "__main__":
        continue_training_generalist()
else:
    print('no gaze reliability vaying training')

In [ ]:
# --- Configuration ---
LOG_FILE = "log_generalist.pkl"
BIN_SIZE = 1000  # Group steps into bins for smoother plotting
ROLLING_WINDOW = 50 # Rolling average window size

# --- 1. Load Data ---
try:
    df = pd.read_pickle(LOG_FILE)
    print(f"Successfully loaded {len(df)} steps from '{LOG_FILE}'")
except FileNotFoundError:
    print(f"Error: Could not find '{LOG_FILE}'. Make sure you ran the training first.")
    df = pd.DataFrame()

if not df.empty:
    # --- 2. Pre-process Data ---
    
    # Create bins for plotting (e.g., every 1000 steps)
    df['step_bin'] = (df['step'] // BIN_SIZE) * BIN_SIZE
    
    # Identify successful pulls (Coordination Reward > 5.0)
    df['A_successful_pull'] = (df['A_action_pull'] == 1) & (df['A_reward'] > 5.0)
    df['B_successful_pull'] = (df['B_action_pull'] == 1) & (df['B_reward'] > 5.0)
    
    # Identify attempted pulls (Any pull action)
    df['A_attempted_pull'] = (df['A_action_pull'] == 1)
    df['B_attempted_pull'] = (df['B_action_pull'] == 1)

    # --- 3. Aggregate by Bin ---
    df_agg = df.groupby('step_bin').agg({
        'A_successful_pull': 'sum',
        'B_successful_pull': 'sum',
        'A_attempted_pull': 'sum',
        'B_attempted_pull': 'sum',
        'A_action_gaze': 'mean',  # Average gaze action (rate) per bin
        'B_action_gaze': 'mean',
        'reliability': 'mean'     # Average reliability for this bin
    }).reset_index()

    # --- 4. Calculate Metrics ---
    
    # Efficiency: Success / Attempts
    df_agg['A_efficiency'] = df_agg['A_successful_pull'] / df_agg['A_attempted_pull'].replace(0, 1)
    df_agg['B_efficiency'] = df_agg['B_successful_pull'] / df_agg['B_attempted_pull'].replace(0, 1)
    
    # Smooth the lines
    df_agg['A_eff_smooth'] = df_agg['A_efficiency'].rolling(window=ROLLING_WINDOW).mean()
    df_agg['B_eff_smooth'] = df_agg['B_efficiency'].rolling(window=ROLLING_WINDOW).mean()
    df_agg['A_gaze_smooth'] = df_agg['A_action_gaze'].rolling(window=ROLLING_WINDOW).mean()
    df_agg['B_gaze_smooth'] = df_agg['B_action_gaze'].rolling(window=ROLLING_WINDOW).mean()

    # --- 5. Calculate IPI (Inter-Pull Interval) separately ---
    # (IPI needs step-by-step differences, not bins)
    df_success_A = df[df['A_successful_pull']].copy()
    df_success_B = df[df['B_successful_pull']].copy()
    
    df_success_A['IPI'] = df_success_A['step'].diff()
    df_success_B['IPI'] = df_success_B['step'].diff()
    
    # Create a rolling average for IPI
    df_success_A['IPI_smooth'] = df_success_A['IPI'].rolling(window=50).mean()
    df_success_B['IPI_smooth'] = df_success_B['IPI'].rolling(window=50).mean()

    # --- PLOTTING ---
    sns.set(style="darkgrid")
    fig, axes = plt.subplots(3, 1, figsize=(16, 18), sharex=False)

    # --- Plot 1: Pull Efficiency ---
    sns.lineplot(ax=axes[0], data=df_agg, x='step_bin', y='A_eff_smooth', label='Agent A Efficiency', color='#0072B2')
    sns.lineplot(ax=axes[0], data=df_agg, x='step_bin', y='B_eff_smooth', label='Agent B Efficiency', color='#D55E00')
    
    # Overlay Reliability (Secondary Axis)
    ax0_twin = axes[0].twinx()
    sns.lineplot(ax=ax0_twin, data=df_agg, x='step_bin', y='reliability', color='gray', alpha=0.3, label='Gaze Reliability', linestyle='--')
    ax0_twin.set_ylim(0, 1.1)
    ax0_twin.set_ylabel('Gaze Reliability', color='gray')
    
    axes[0].set_title('Pull Efficiency (with Reliability Context)', fontsize=16)
    axes[0].set_ylabel('Efficiency (Success/Attempts)', fontsize=12)
    axes[0].set_ylim(0, 1.0)
    axes[0].legend(loc='upper left')

    # --- Plot 2: Inter-Pull Interval (IPI) ---
    sns.lineplot(ax=axes[1], data=df_success_A, x='step', y='IPI_smooth', label='Agent A IPI', color='#0072B2')
    sns.lineplot(ax=axes[1], data=df_success_B, x='step', y='IPI_smooth', label='Agent B IPI', color='#D55E00')
    axes[1].axhline(y=65, color='gray', linestyle=':', label='Optimal (approx 65)')
    
    axes[1].set_title('Inter-Pull Interval (IPI)', fontsize=16)
    axes[1].set_ylabel('Steps Between Successful Pulls', fontsize=12)
    axes[1].set_ylim(0, 300) # Adjust based on your data range
    axes[1].legend(loc='upper left')

    # --- Plot 3: Gaze Rate ---
    sns.lineplot(ax=axes[2], data=df_agg, x='step_bin', y='A_gaze_smooth', label='Agent A Gaze Rate', color='#0072B2')
    sns.lineplot(ax=axes[2], data=df_agg, x='step_bin', y='B_gaze_smooth', label='Agent B Gaze Rate', color='#D55E00')
    
    # Overlay Reliability (Secondary Axis)
    ax2_twin = axes[2].twinx()
    sns.lineplot(ax=ax2_twin, data=df_agg, x='step_bin', y='reliability', color='gray', alpha=0.3, label='Gaze Reliability', linestyle='--')
    ax2_twin.set_ylim(0, 1.1)
    ax2_twin.set_ylabel('Gaze Reliability', color='gray')

    axes[2].set_title('Gaze Rate Adaptation', fontsize=16)
    axes[2].set_ylabel('Gaze Probability (Avg per bin)', fontsize=12)
    axes[2].set_xlabel('Training Step', fontsize=14)
    axes[2].legend(loc='upper left')

    plt.tight_layout()
    plt.savefig("generalist_training_analysis.pdf")
    print("Saved plot to 'generalist_training_analysis.pdf'")
    plt.show()

## use trained data to test the hypothesis
### helper functions

In [ ]:
#################################################################
# 3. HELPER FUNCTIONS (NOW INCLUDES STOCHASTIC SAMPLING)
#################################################################

# --- 1. Define the Helper Function First ---
def sample_actions_for_testing(logits_tuple):
    """
    Takes logits from the Actor and STOCHASTICALLY samples
    one action from each category.
    """
    move_logits, pull_logits, gaze_logits = logits_tuple
    
    # --- Move Action ---
    move_dist = Categorical(logits=move_logits)
    move_action = move_dist.sample() 
    
    # --- Pull Action ---
    pull_dist = Categorical(logits=pull_logits)
    pull_action = pull_dist.sample()
    
    # --- Gaze Action ---
    gaze_dist = Categorical(logits=gaze_logits)
    gaze_action = gaze_dist.sample()
    
    actions_dict = {
        'move': move_action.item(),
        'pull': pull_action.item(),
        'gaze': gaze_action.item()
    }
    
    return actions_dict

### testing functions with the gaze reliability change conditions

In [ ]:
#################################################################
# 8. ANALYSIS: PARAMETRIC SWEEP (Reliability vs. Gaze Rate)
#################################################################

def run_reliability_sweep():
    print("--- Starting Parametric Sweep: Reliability vs. Gaze Rate ---")
    
    # --- 1. Setup ---
    RELIABILITIES = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
    STEPS_PER_COND = 20000 # Enough steps to get a stable average
    TARGET_WINDOW = 10
    
    # Load the Generalist Models
    agent_A = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_B = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    
    try:
        agent_A.load_state_dict(torch.load("agent_A_generalist.pth"))
        agent_B.load_state_dict(torch.load("agent_B_generalist.pth"))
        print("Loaded Generalist Experts.")
    except FileNotFoundError:
        print("Error: Generalist models not found.")
        return

    agent_A.eval()
    agent_B.eval()
    
    env = CurriculumEnv()
    env.set_training_phase("COOP_FULL", sync_window=TARGET_WINDOW)
    
    results = []

    # --- 2. The Loop ---
    for rel in RELIABILITIES:
        print(f"Testing Reliability: {rel*100:.0f}%...")
        
        state_A_dict, state_B_dict = env.reset()
        public_A = preprocess_state(state_A_dict)
        public_B = preprocess_state(state_B_dict)
        hidden_A = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
        hidden_B = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
        private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
        private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
        
        gaze_count_A = 0
        gaze_count_B = 0
        
        with torch.no_grad():
            for _ in range(STEPS_PER_COND):
                
                # Masks
                if state_A_dict['can_pull']: mask_A = torch.tensor([[0., 0.]], device=device).float()
                else: mask_A = torch.tensor([[0., -1e9]], device=device).float()
                if state_B_dict['can_pull']: mask_B = torch.tensor([[0., 0.]], device=device).float()
                else: mask_B = torch.tensor([[0., -1e9]], device=device).float()

                # Agents
                (logits_A), _, next_h_A = agent_A(public_A, private_A, hidden_A, mask_A)
                actions_A = sample_actions_for_testing(logits_A)
                
                (logits_B), _, next_h_B = agent_B(public_B, private_B, hidden_B, mask_B)
                actions_B = sample_actions_for_testing(logits_B)
                
                # Env
                next_A_dict, next_B_dict, _, _, _ = env.step(actions_A, actions_B)
                
                # Count Gazes
                gaze_count_A += actions_A['gaze']
                gaze_count_B += actions_B['gaze']
                
                # --- APPLY THE RELIABILITY MASK (The Test) ---
                pB_can = torch.tensor([[1.0 if next_B_dict['can_pull'] else 0.0]], device=device).float()
                pB_time = torch.tensor([[min(env.B_time_since_pull / env.current_sync_window, 1.0)]], device=device).float()
                pB_drink = torch.tensor([[1.0 if next_B_dict['self_is_drinking'] else 0.0]], device=device).float()
                pB_cool = torch.tensor([[1.0 if next_B_dict['self_on_cooldown'] else 0.0]], device=device).float()
                
                pA_can = torch.tensor([[1.0 if next_A_dict['can_pull'] else 0.0]], device=device).float()
                pA_time = torch.tensor([[min(env.A_time_since_pull / env.current_sync_window, 1.0)]], device=device).float()
                pA_drink = torch.tensor([[1.0 if next_A_dict['self_is_drinking'] else 0.0]], device=device).float()
                pA_cool = torch.tensor([[1.0 if next_A_dict['self_on_cooldown'] else 0.0]], device=device).float()

                # Die Roll based on CURRENT TEST CONDITION 'rel'
                mask_val_A = (torch.rand(1, device=device) < rel).float()
                mask_val_B = (torch.rand(1, device=device) < rel).float()

                if actions_A['gaze'] == 1:
                    private_A = torch.cat([pB_can, pB_time, pB_drink, pB_cool], dim=-1) * mask_val_A
                else:
                    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
                    
                if actions_B['gaze'] == 1:
                    private_B = torch.cat([pA_can, pA_time, pA_drink, pA_cool], dim=-1) * mask_val_B
                else:
                    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
                
                # Update
                public_A = preprocess_state(next_A_dict)
                public_B = preprocess_state(next_B_dict)
                state_A_dict = next_A_dict
                state_B_dict = next_B_dict
                hidden_A = next_h_A
                hidden_B = next_h_B
        
        # Store results
        results.append({
            'reliability': rel,
            'gaze_rate_A': gaze_count_A / STEPS_PER_COND,
            'gaze_rate_B': gaze_count_B / STEPS_PER_COND
        })

    # --- 3. Plotting ---
    df_res = pd.DataFrame(results)
    
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=df_res, x='reliability', y='gaze_rate_A', label='Agent A (Leader)', marker='o', linewidth=3, color='#0072B2')
    sns.lineplot(data=df_res, x='reliability', y='gaze_rate_B', label='Agent B (Follower)', marker='o', linewidth=3, color='#D55E00')
    
    plt.title("Cognitive Flexibility: Gaze Adaptation to Uncertainty", fontsize=16)
    plt.xlabel("Sensory Reliability (Information Quality)", fontsize=14)
    plt.ylabel("Gaze Probability (Information Seeking)", fontsize=14)
    plt.ylim(0, 1.0)
    plt.gca().invert_xaxis() # Invert X so "Hard" (Low Rel) is on the right, or keep normal.
    # Usually, we want Low Reliability -> High Gaze.
    
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.savefig("final_gaze_adaptation_curve.pdf")
    print("Saved 'final_gaze_adaptation_curve.pdf'")
    plt.show()
    
    print("\n--- Results Table ---")
    print(df_res)

if __name__ == "__main__":
    run_reliability_sweep()

In [ ]:
np.unique(df['A_reward'])

In [ ]:
np.unique(df['B_reward'])